<a href="https://colab.research.google.com/github/Thampi-hub/Springboard_RT/blob/main/Capstone_2/2_DataWrangling/readData.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [83]:
# Import libs
import pandas as pd
import numpy as np
from datetime import datetime

In [84]:
### Read in JSON files (Category names):

# ca_json = pd.read_json("./Kaggle_data/CA_category_id.json")["items"].to_dict()
# us_json = pd.read_json("./Kaggle_data/US_category_id.json")["items"].to_dict()
#       --------- OR ---------
url_ca_json = "https://raw.githubusercontent.com/Thampi-hub/Springboard_RT/refs/heads/main/Capstone_2/2_DataWrangling/Kaggle_data/CA_category_id.json"
url_us_json = "https://raw.githubusercontent.com/Thampi-hub/Springboard_RT/refs/heads/main/Capstone_2/2_DataWrangling/Kaggle_data/US_category_id.json"
ca_json = pd.read_json(url_ca_json)["items"].to_dict()
us_json = pd.read_json(url_us_json)["items"].to_dict()

cat_id  = []
cat_val = []
for idx in ca_json.values():
    cat_id.append( int(idx["id"]) )
    cat_val.append(idx["snippet"]["title"])
for idx in us_json.values():
    cat_id.append( int(idx["id"]) )
    cat_val.append(idx["snippet"]["title"])

category_mapping = pd.DataFrame({'category_id':cat_id, 'category':cat_val }).drop_duplicates()

In [85]:
### Read in CSV files (Core data files):

# ca_csv = pd.read_csv("./Kaggle_data/CAvideos.csv")
# us_csv = pd.read_csv("./Kaggle_data/USvideos.csv")
#       --------- OR ---------
url_ca_csv = "https://raw.githubusercontent.com/Thampi-hub/Springboard_RT/refs/heads/main/Capstone_2/2_DataWrangling/Kaggle_data/CAvideos.csv"
url_us_csv = "https://raw.githubusercontent.com/Thampi-hub/Springboard_RT/refs/heads/main/Capstone_2/2_DataWrangling/Kaggle_data/USvideos.csv"
ca_csv = pd.read_csv(url_ca_csv)
us_csv = pd.read_csv(url_us_csv)

ca_csv['country'] = "Canada"
us_csv['country'] = "USA"

allCSV = pd.concat([ca_csv,us_csv], axis=0)
print("CSV dimensions: ", ca_csv.shape, " + ", us_csv.shape, " = ", allCSV.shape)


CSV dimensions:  (40881, 17)  +  (40949, 17)  =  (81830, 17)


In [104]:
#Merge all data:
allData = pd.merge(allCSV, category_mapping, on="category_id", how="left")

print("Dimensions: ", allData.shape, "\n\n")
print(allData.columns.values,"\n")
print(allData.dtypes)

Dimensions:  (81830, 18) 


['video_id' 'trending_date' 'title' 'channel_title' 'category_id'
 'publish_time' 'tags' 'views' 'likes' 'dislikes' 'comment_count'
 'thumbnail_link' 'comments_disabled' 'ratings_disabled'
 'video_error_or_removed' 'description' 'country' 'category'] 

video_id                  object
trending_date             object
title                     object
channel_title             object
category_id                int64
publish_time              object
tags                      object
views                      int64
likes                      int64
dislikes                   int64
comment_count              int64
thumbnail_link            object
comments_disabled           bool
ratings_disabled            bool
video_error_or_removed      bool
description               object
country                   object
category                  object
dtype: object


In [105]:
### Convert to correct type:

allData['trending_date'] = pd.to_datetime(allData['trending_date'], format="%y.%d.%m")

allData['publish_datetime']  = pd.to_datetime(allData['publish_time'], utc=True)
allData['publish_date']  = allData['publish_datetime'].dt.date
allData['publish_time']  = allData['publish_datetime'].dt.time


In [112]:
### Do we need 'video_error_or_removed' column?

## Deleted videos, based on title:
deleted_id = allData[allData.title=="Deleted video"]["video_id"].unique()

## Using flag 'video_error_or_removed':
err_rem_id = allData[allData.video_error_or_removed]["video_id"].unique()

## Videos uniquely flagged by 'video_error_or_removed':
uniq_err_id = list(set(err_rem_id) - set(deleted_id)) #['KjSSU1trCug','0JqXITEsMy8','q8v9MvManKE','BEePFpC9qG8','1Aoc-cd9eYs','RK_B4Ez4_5Q']
allData[allData.video_id.isin(uniq_err_id)].sort_values(["video_id","country","publish_datetime","trending_date"]).head(10)


Index(['video_id', 'publish_datetime', 'trending_date', 'title',
       'channel_title', 'category', 'category_id', 'country', 'description',
       'publish_date', 'publish_time', 'likes', 'dislikes', 'comment_count',
       'views', 'tags', 'thumbnail_link', 'comments_disabled',
       'ratings_disabled', 'video_error_or_removed'],
      dtype='object')

In [144]:
### Filter data for only required data (ID columns: video_id, country)

## Row-filters: latest publish_datetime, followed by earliest trending date.
filtered_idx = allData[allData.title!="Deleted video"].\
                sort_values(["video_id","country","publish_datetime","trending_date"]).\
                groupby(["video_id","country"])["publish_datetime"].\
                idxmax().values

## Col-filter: Not required for our analysis
drop_cols = ["thumbnail_link","video_error_or_removed"]

## Apply both filters
filtrData = allData.loc[allData.index.isin(filtered_idx), ~allData.columns.isin(drop_cols)]
filtrData.columns

Index(['video_id', 'publish_datetime', 'trending_date', 'title',
       'channel_title', 'category', 'category_id', 'country', 'description',
       'publish_date', 'publish_time', 'likes', 'dislikes', 'comment_count',
       'views', 'tags', 'comments_disabled', 'ratings_disabled'],
      dtype='object')

In [145]:
# Reorder columns: ID vars, followed by features
id_cols = ["video_id","publish_datetime","trending_date","title","channel_title","category","category_id","country","description","publish_date","publish_time","views","likes","dislikes","comment_count"]
new_col_order = id_cols + filtrData.columns[ ~filtrData.columns.isin(id_cols)].tolist()
filtrData = filtrData[new_col_order]

filtrData.columns

Index(['video_id', 'publish_datetime', 'trending_date', 'title',
       'channel_title', 'category', 'category_id', 'country', 'description',
       'publish_date', 'publish_time', 'views', 'likes', 'dislikes',
       'comment_count', 'tags', 'comments_disabled', 'ratings_disabled'],
      dtype='object')

In [146]:
### Deriving new features:

## No. of tags:
filtrData['n_tags'] = filtrData['tags'].apply(lambda x: len(x.split("|")) )

## No. of Repeats:
trend_rep_DF = allData.groupby(['video_id','country','publish_datetime']).\
                size().rename("trend_rep")
filtrData = pd.merge(filtrData, trend_rep_DF, on=["video_id","country","publish_datetime"], how="left", validate="one_to_one")
filtrData.head()


,video_id,publish_datetime,trending_date,title,channel_title,category,category_id,country,description,publish_date,publish_time,views,likes,dislikes,comment_count,tags,comments_disabled,ratings_disabled,n_tags,trend_rep
19411,1Aoc-cd9eYs,2018-05-02 16:02:35+00:00,2018-05-05,Cobra Kai Ep 2 - Strike First - The Karate Kid...,Cobra Kai,Entertainment,24,Canada,Present day Daniel LaRusso lives a charmed lif...,2018-05-02,16:02:35,673534,27216,930,2630,"Cobra Kai|""Karate Kid""|""YouTube Red Original S...",False,False,20,2
30156,1Aoc-cd9eYs,2018-05-02 16:02:35+00:00,2018-05-05,Cobra Kai Ep 2 - Strike First - The Karate Kid...,Cobra Kai,Entertainment,24,USA,Present day Daniel LaRusso lives a charmed lif...,2018-05-02,16:02:35,833027,28696,3766,5033,"Cobra Kai|""Karate Kid""|""YouTube Red Original S...",False,False,20,7
